# Multi-label ticket classification with DistilBERT

This notebook runs the **full path** from raw CSV to fine-tuned model: **cleaning → multi-label targets → train/val/test → Hugging Face tokenization → DistilBERT fine-tuning → final test metrics**.

The markdown before each code block focuses on **why** we take the step (risk, objective, or production constraint), not a line-by-line narration.


### Why start with imports and project root on `sys.path`?

You want **one reproducible environment**: stable library versions, and imports that resolve whether Jupyter’s working directory is the repo root or a subfolder. Putting the project root on `sys.path` lets you reuse shared config (paths, column names) without copying magic strings across notebooks—reducing “works on my machine” drift when the dataset path or leakage column list changes.


In [1]:
import json
import warnings
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Shared constants only (single source of truth in ml_pipeline/config.py)
from ml_pipeline.config import (
    DEFAULT_DATA_PATH,
    LEAKAGE_COLUMNS,
    REQUIRED_RAW_COLUMNS,
    TEXT_COLUMN,
    SplitConfig,
    TransformerTrainConfig,
)
from ml_pipeline.metrics import multilabel_metric_bundle


### Why centralize knobs (sample size, seeds, output dir) in one place?

**Sample limits** (`NROWS`) let you iterate quickly on a laptop before paying the cost of the full 200k rows. A fixed **random seed** keeps splits and any stochastic behavior comparable across runs. A dedicated **output directory** avoids overwriting checkpoints accidentally and mirrors how you would version artifacts in production (model registry / object storage).


In [2]:
# --- Run configuration ---
NROWS = 25_000  # None = entire CSV
DATA_PATH = DEFAULT_DATA_PATH
SEED = 42
split_cfg = SplitConfig(test_size=0.15, val_size=0.15, random_state=SEED)

OUTPUT_DIR = ROOT / "outputs" / "multilabel_distilbert_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_cfg = TransformerTrainConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=2,
    max_length=256,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    fp16=torch.cuda.is_available(),
)


### Why validate the schema immediately after loading?

Downstream code assumes specific columns exist (`issue_description`, `category`, `priority`). Failing **early** with a clear error is cheaper than debugging a `KeyError` deep inside tokenization or training. It also documents the **contract** your pipeline expects from upstream data producers.


In [3]:
df_raw = pd.read_csv(DATA_PATH, nrows=NROWS, low_memory=False)
missing = [c for c in REQUIRED_RAW_COLUMNS if c not in df_raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
print("Loaded shape:", df_raw.shape)
df_raw.head(2)


Loaded shape: (25000, 30)


,ticket_id,customer_name,customer_email,product,category,issue_description,resolution_notes,priority,status,channel,...,ticket_resolved_date,escalated,sla_breached,operating_system,browser,payment_method,language,preferred_contact_time,issue_complexity_score,customer_segment
0,1,Patricia Smith,patricia.smith760@outlook.com,Web Portal,Account Suspension,The payment was deducted from my bank account ...,Data synchronization restored after backend se...,Urgent,Open,Email,...,2023-05-20,No,Yes,MacOS,Edge,PayPal,French,Afternoon,4,Small Business
1,2,Patricia Williams,patricia.williams390@gmail.com,Mobile App,Performance Issue,I found a bug in the latest update affecting r...,Provided step-by-step troubleshooting instruct...,Urgent,Closed,Email,...,2024-01-19,Yes,Yes,Windows,Firefox,PayPal,English,Afternoon,2,Small Business


### Why normalize ticket text before modeling?

Raw tickets mix **casing**, **HTML**, **URLs**, and **email-like tokens**. Without normalization, the model wastes capacity on superficial variation and may **memorize PII-shaped strings** instead of semantics. A shared cleaning function also ensures **training and inference follow the same rules**—otherwise you get silent train/serve skew and confusing regressions after deployment.


In [4]:
import re

EMAIL_RE = re.compile(r"\b[\w.-]+@[\w.-]+\.\w+\b")
URL_RE = re.compile(r"https?://\S+|www\.\S+")
HTML_RE = re.compile(r"<.*?>")
EXTRA_SPACE_RE = re.compile(r"\s+")
ALLOWED_CHARS_RE = re.compile(r"[^a-zA-Z\s.,!?]")


def clean_ticket_text(text) -> str:
    if text is None or (isinstance(text, float) and str(text) == "nan"):
        return ""
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()
    text = HTML_RE.sub(" ", text)
    text = EMAIL_RE.sub(" <email> ", text)
    text = URL_RE.sub(" <url> ", text)
    text = ALLOWED_CHARS_RE.sub(" ", text)
    text = EXTRA_SPACE_RE.sub(" ", text).strip()
    return text


clean_ticket_text("Contact me at A@B.com re: https://x.com/y !!!")


'contact me at email re url !!!'

### Why define multi-label targets as `category:*` and `priority:*` tags?

Your business outcome is often **both** “what kind of problem” and “how urgent”—a **multi-label** setup (several tags active per ticket) matches that better than predicting only one head. Prefixing tags (`category:` / `priority:`) keeps namespaces separate so a string like `High` does not collide across fields. We then binarize with `MultiLabelBinarizer` **after** splitting so the label vocabulary is learned from **training data only**, avoiding information leak from validation or test.


In [5]:
def _tag_category(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    s = str(val).strip()
    if not s or s.lower() in {"", "nan", "none", "null", "na", "n/a"}:
        return None
    return f"category:{s}"


def _tag_priority(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    s = str(val).strip()
    if s.lower() in {"", "nan", "none", "null", "na", "n/a"}:
        return None
    return f"priority:{s}"


def row_to_label_set(row: pd.Series) -> list[str]:
    tags = []
    c = _tag_category(row.get("category"))
    p = _tag_priority(row.get("priority"))
    if c:
        tags.append(c)
    if p:
        tags.append(p)
    return tags


df = df_raw.copy()
df[TEXT_COLUMN] = df[TEXT_COLUMN].replace("", np.nan)
df["label_set"] = df.apply(row_to_label_set, axis=1)
df["clean_text"] = df[TEXT_COLUMN].map(clean_ticket_text)
valid = df["clean_text"].str.len() > 0
has_label = df["label_set"].map(len) > 0
df = df.loc[valid & has_label].reset_index(drop=True)

df["_label_key"] = df["label_set"].map(lambda s: tuple(sorted(s)))
df = df.drop_duplicates(subset=["clean_text", "_label_key"]).reset_index(drop=True)
df = df.drop(columns=["_label_key"])

print("After cleaning + dedup:", df.shape)
df[["clean_text", "label_set"]].head(3)


After cleaning + dedup: (400, 32)


,clean_text,label_set
0,the payment was deducted from my bank account ...,"[category:Account Suspension, priority:Urgent]"
1,i found a bug in the latest update affecting r...,"[category:Performance Issue, priority:Urgent]"
2,the application crashes whenever i try to uplo...,"[category:Performance Issue, priority:Medium]"


### Why drop resolution / SLA style columns even if they exist?

If those fields are recorded **after** or **because of** handling, using them as inputs to predict triage labels is **outcome leakage**: the model sees the future. That inflates offline metrics and fails in real routing where those fields are missing or different. Dropping them enforces a **causal ordering**: inputs must be plausibly available **at ticket creation time** for the prediction task you care about.


In [6]:
leak_present = LEAKAGE_COLUMNS.intersection(df.columns)
if leak_present:
    warnings.warn("Dropping leakage columns: " + ", ".join(sorted(leak_present)))
    df = df.drop(columns=list(leak_present), errors="ignore")

df[TEXT_COLUMN] = df["clean_text"]
print("Leakage columns removed; modeling text column set to cleaned text.")


Leakage columns removed; modeling text column set to cleaned text.


/tmp/ipykernel_1381308/3467291867.py:3: UserWarning: Dropping leakage columns: customer_satisfaction_score, escalated, first_response_time_hours, resolution_notes, resolution_time_hours, sla_breached, status, ticket_resolved_date
  warnings.warn("Dropping leakage columns: " + ", ".join(sorted(leak_present)))


### Why three splits (train / validation / test) and stratify on `category`?

**Validation** is for hyperparameter and early-stopping decisions; **test** should be touched **once** for an unbiased generalization estimate. Pure random splits are fine, but stratifying on **category** (a single-label proxy) keeps rare classes from disappearing entirely from train or validation when possible. `MultiLabelBinarizer` is **fit on train only**; validation/test rows whose tag combinations never appeared in train are dropped so `transform` never sees unknown classes.


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

strat_col = "category" if "category" in df.columns else None
idx = np.arange(len(df))
strat = df[strat_col] if strat_col else None
if strat is not None and strat.value_counts().min() < 2:
    strat = None

idx_train, idx_temp = train_test_split(
    idx,
    test_size=split_cfg.test_size + split_cfg.val_size,
    random_state=split_cfg.random_state,
    stratify=strat,
)
strat_temp = df.loc[idx_temp, strat_col] if strat_col else None
if strat_temp is not None and strat_temp.value_counts().min() < 2:
    strat_temp = None

rel_test = split_cfg.test_size / (split_cfg.test_size + split_cfg.val_size)
idx_val, idx_test = train_test_split(
    idx_temp,
    test_size=rel_test,
    random_state=split_cfg.random_state,
    stratify=strat_temp,
)

train_df = df.iloc[idx_train].reset_index(drop=True)
val_df = df.iloc[idx_val].reset_index(drop=True)
test_df = df.iloc[idx_test].reset_index(drop=True)

mlb = MultiLabelBinarizer(sparse_output=False)
Y_train = mlb.fit_transform(train_df["label_set"])
classes_set = set(mlb.classes_)


def _only_known(ls):
    return bool(ls) and all(t in classes_set for t in ls)


val_df = val_df.loc[val_df["label_set"].map(_only_known)].reset_index(drop=True)
test_df = test_df.loc[test_df["label_set"].map(_only_known)].reset_index(drop=True)
Y_val = mlb.transform(val_df["label_set"])
Y_test = mlb.transform(test_df["label_set"])

print("train", len(train_df), "val", len(val_df), "test", len(test_df))
print("num_labels", len(mlb.classes_))


train 280 val 60 test 60
num_labels 14


### Why Hugging Face tokenization *without* padding inside `Dataset.map`?

Padding every example to `max_length` in advance **multiplies memory** (especially on large CSVs). For transformer training, you typically **tokenize with truncation only**, then let a **`DataCollatorWithPadding`** pad **per batch** to the longest sequence in that batch. That keeps activations smaller and is the standard HF + PyTorch pattern for variable-length text.


In [8]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(train_cfg.model_name)

texts_train = [clean_ticket_text(t) for t in train_df["clean_text"].tolist()]
texts_val = [clean_ticket_text(t) for t in val_df["clean_text"].tolist()]

labels_train = [row.astype(float).tolist() for row in Y_train.astype(float)]
labels_val = [row.astype(float).tolist() for row in Y_val.astype(float)]

train_ds = Dataset.from_dict({"text": texts_train, "labels": labels_train})
val_ds = Dataset.from_dict({"text": texts_val, "labels": labels_val})


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=train_cfg.max_length,
        padding=False,
    )


train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

print(train_ds)
print("Sample keys:", train_ds[0].keys())


Map:   0%|          | 0/280 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 280
})
Sample keys: dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])


### Why DistilBERT with `problem_type="multi_label_classification"`?

DistilBERT is a lighter cousin of BERT: faster iteration with still strong sentence-level behavior—useful for ticket-length text. For **multi-label** targets, each label is **independent** given the sentence; HF uses a **sigmoid + BCEWithLogitsLoss** setup instead of softmax cross-entropy. That matches your binary vector `Y` where **more than one** label can be active per row.


In [9]:
from transformers import AutoModelForSequenceClassification

num_labels = Y_train.shape[1]
model = AutoModelForSequenceClassification.from_pretrained(
    train_cfg.model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification",
)
model.config.problem_type


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


'multi_label_classification'

### Why `Trainer` + validation metrics + *no* test set during `train`?

`Trainer` orchestrates the training loop, checkpointing, and mixed precision when available. We evaluate on **`val_ds`** so we can pick a reasonable stopping point and compare epochs **without** peeking at the test set—using test for that would **optimistically bias** reported performance. **Test** is reserved for a **single** post-training evaluation cell.


In [10]:
import inspect
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.metrics import f1_score, hamming_loss

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")
THRESH = train_cfg.prediction_threshold


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs >= THRESH).astype(int)
    labels = labels.astype(int)
    return {
        "f1_micro": float(f1_score(labels, preds, average="micro", zero_division=0)),
        "f1_macro": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "hamming_loss": float(hamming_loss(labels, preds)),
    }


supported = set(inspect.signature(TrainingArguments.__init__).parameters)
kwargs = {
    "output_dir": train_cfg.output_dir,
    "learning_rate": train_cfg.learning_rate,
    "weight_decay": train_cfg.weight_decay,
    "warmup_ratio": train_cfg.warmup_ratio,
    "num_train_epochs": train_cfg.num_train_epochs,
    "per_device_train_batch_size": train_cfg.per_device_train_batch_size,
    "per_device_eval_batch_size": train_cfg.per_device_eval_batch_size,
    "save_strategy": "epoch",
    "logging_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1_micro",
    "greater_is_better": True,
    "save_total_limit": 2,
    "report_to": "none",
}
if train_cfg.fp16 and torch.cuda.is_available():
    kwargs["fp16"] = True
if "evaluation_strategy" in supported:
    kwargs["evaluation_strategy"] = "epoch"
elif "eval_strategy" in supported:
    kwargs["eval_strategy"] = "epoch"
else:
    kwargs.pop("load_best_model_at_end", None)
    kwargs.pop("metric_for_best_model", None)
    kwargs.pop("greater_is_better", None)

kwargs = {k: v for k, v in kwargs.items() if k in supported}
training_args = TrainingArguments(**kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

### Why save model, tokenizer, and `label_names.json` together?

At inference time you need **three** aligned pieces: tokenizer (to reproduce input IDs), model weights (sigmoid logits), and the **order of labels** (each column of `Y` maps to an index in the head). Saving `label_names.json` next to the checkpoint prevents silent mismatches when someone loads the model months later or in another service.


In [ ]:
trainer.train()
trainer.save_model(train_cfg.output_dir)
tokenizer.save_pretrained(train_cfg.output_dir)

label_path = Path(train_cfg.output_dir) / "label_names.json"
label_path.write_text(json.dumps(mlb.classes_.tolist(), indent=2), encoding="utf-8")
print("Saved to", train_cfg.output_dir)


### Why evaluate on the test set only *after* training finishes?

The test set estimates **generalization to unseen data**. If you repeatedly tune on test (or use it inside `Trainer`), metrics become **overfit to that split** and stop answering “how will this behave on next week’s tickets?”. One final pass here keeps the estimate honest; use validation for everything iterative.


In [ ]:
texts_test = [clean_ticket_text(t) for t in test_df["clean_text"].tolist()]

model.eval()
device = next(model.parameters()).device
batch_size = 16
all_probs = []

for start in range(0, len(texts_test), batch_size):
    batch = texts_test[start : start + batch_size]
    enc = tokenizer(
        batch,
        truncation=True,
        padding=True,
        max_length=train_cfg.max_length,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
        all_probs.append(torch.sigmoid(logits).cpu().numpy())

probs = np.vstack(all_probs)
preds = (probs >= THRESH).astype(int)

print("Test metrics:", multilabel_metric_bundle(Y_test, preds, y_scores=probs, threshold=THRESH))
